# Final PP pricing solution — computation notebook

This notebook builds the team's final price file. It fits no new model. It combines saved team outputs:

- **Core:** Maksim's frozen premium — fair value × 1.25, never below the API base price (`experiments/fair_value_model/`; the engine copy is in `main/fair_value_engine/`).
- **Improvement:** Chris's order-flow loading (`experiments/order_flow_book/`), only if it passes the rule committed in `README.md`
  before any scoring.
- **Risk rule:** Maksim's exposure cap per direction. It is reported, not priced.

Run it from top to bottom. It writes `outputs/`. The plain-English report is `final_report.ipynb` → `final_report.html`.

In [1]:
import hashlib
import json
import subprocess
import sys
from pathlib import Path

import numpy as np
import pandas as pd

HERE = Path.cwd()
sys.path.insert(0, str(HERE))
import solution as s

DATA = s.find_data_root(HERE)  # the folder that holds data/
sys.path.insert(0, str(s.ENGINE))  # Maksim's engine in main/fair_value_engine
from src.evaluate import conservative_weight, evaluate_scenarios, gini_with_interval, paired_profit_interval

OUT = HERE / "outputs"
OUT.mkdir(exist_ok=True)
pd.set_option("display.width", 200, "display.max_columns", 40)


def sha256(path: Path) -> str:
    return hashlib.sha256(path.read_bytes()).hexdigest()


PATHS = {
    "panel": s.EVIDENCE / "fair_value_model/results/walk_forward_orders_priced.parquet",
    "training": DATA / "data/with_base_premium/pp_transactions_training.csv",
    "validation": DATA / "data/validation/pp_transactions_validation.csv",
    "frozen": s.ENGINE / "outputs/validation_recommendations.csv",
    "rebuilt_base": s.EVIDENCE / "hybrid_base_multiplier/outputs/validation_premiums.csv",
}
frozen_record = json.loads((s.ENGINE / "outputs/FROZEN.json").read_text())
frozen_ok = sha256(PATHS["frozen"]) == frozen_record["sha256"]
assert frozen_ok, "fair_value_engine/outputs/validation_recommendations.csv differs from its FROZEN.json"
commit = subprocess.run(["git", "-C", str(DATA), "rev-parse", "--short", "HEAD"],
                        capture_output=True, text=True).stdout.strip()
print("data root:", DATA.name, "| commit:", commit, "| frozen file matches FROZEN.json:", frozen_ok)

data root: DBA5102_Jasper_Vault_Assignment | commit: b7ac940 | frozen file matches FROZEN.json: True


## A. Data

| Name | File | Use |
|---|---|---|
| panel | `evidence/fair_value_model/results/walk_forward_orders_priced.parquet` | 7,150 training orders with outcomes. Maksim's price for each month comes from a model fitted on earlier months only. |
| training | `data/with_base_premium/pp_transactions_training.csv` | All 7,204 training orders, to rebuild JV's open book. |
| validation | `data/validation/pp_transactions_validation.csv` | The 4,155 blind orders. `closedAt` is dropped on read. |
| frozen | `fair_value_engine/outputs/validation_recommendations.csv` | Maksim's frozen premiums. Its SHA-256 must match `FROZEN.json`. |

In [2]:
panel = pd.read_parquet(PATHS["panel"])
train = s.read_orders(PATHS["training"])
valid = s.read_orders(PATHS["validation"])
frozen = s.read_orders(PATHS["frozen"])

# Maksim's file keeps the original columns as text. Check the non-outcome ones against the validation file.
raw_valid = pd.read_csv(PATHS["validation"], dtype=str, keep_default_na=False)
raw_frozen = pd.read_csv(PATHS["frozen"], dtype=str, keep_default_na=False)
same_columns = [c for c in raw_valid.columns if c not in s.OUTCOME_COLUMNS]
assert raw_frozen[same_columns].equals(raw_valid[same_columns])
assert panel["row_id"].is_unique and panel["row_id"].between(0, len(train) - 1).all()
assert not set(s.OUTCOME_COLUMNS) & set(valid.columns)

print(f"panel {len(panel):,} | training {len(train):,} | validation {len(valid):,} | frozen {len(frozen):,}")
panel.groupby("month").size().rename("orders").to_frame().T

panel 7,150 | training 7,204 | validation 4,155 | frozen 4,155


month,2026-02,2026-03,2026-04,2026-05,2026-06,2026-07
orders,211,266,867,965,2270,2571


## B. The order-flow loading on training orders

JV's open book before each order holds the earlier orders that have not expired (`closedAt` is never used).
`s_add` > 0 means the new order joins the larger side of the book. `s_add` < 0 means it opposes the crowd.
Loading = 1 + 0.3 × max(−s_add, 0) when the book holds at least 1 BTC, else 1.

In [3]:
book_train = s.book_state(train[s.BOOK_COLUMNS], train)   # each order sees earlier, unexpired orders only
panel = panel.join(book_train[["book_btc", "s_add"]], on="row_id")
panel["loading"] = s.orderflow_loading(panel["s_add"], panel["book_btc"])

loaded = panel["loading"] > 1
print(f"loaded share {loaded.mean():.3f} | mean loading on loaded orders {panel.loc[loaded, 'loading'].mean():.4f}")
loading_by_month = panel.assign(loaded=loaded).groupby("month").agg(
    orders=("loading", "size"), loaded_share=("loaded", "mean"),
    mean_loading_on_loaded=("loading", lambda x: x[x > 1].mean()))

book_class = np.select([panel["book_btc"] < 1, panel["s_add"] > 0.2, panel["s_add"] < -0.2],
                       ["book under 1 BTC", "joins the crowd", "opposes the crowd"], "neutral")
book_classes = panel.assign(book_class=book_class).groupby("book_class").agg(
    orders=("row_id", "size"), jv_premium=("premiumAmount_USDC", "sum"),
    payout=("realizedProfit_USDC", "sum"), fair_value=("fair_value_USDC", "sum"))
book_classes["loss_ratio_at_jv_price"] = book_classes["payout"] / book_classes["jv_premium"]
book_classes["payout_over_fair_value"] = book_classes["payout"] / book_classes["fair_value"]
display(loading_by_month.round(3), book_classes.round(3))

loaded share 0.218 | mean loading on loaded orders 1.0770


,orders,loaded_share,mean_loading_on_loaded
month,,,
2026-02,211,0.000,NaN
2026-03,266,0.000,NaN
2026-04,867,0.060,1.072
2026-05,965,0.155,1.076
2026-06,2270,0.244,1.101
2026-07,2571,0.312,1.061


,orders,jv_premium,payout,fair_value,loss_ratio_at_jv_price,payout_over_fair_value
book_class,,,,,,
book under 1 BTC,3676,76079.031,61713.065,65812.795,0.811,0.938
joins the crowd,898,52025.998,35700.202,46421.479,0.686,0.769
neutral,1968,87725.927,96993.791,89650.567,1.106,1.082
opposes the crowd,608,32258.049,36244.995,32502.866,1.124,1.115


## C. Training evidence and the adoption rule

Strategies: **JV** (actual price), **M** (Maksim), **M+L** (M × loading), **Flat-M** (M raised by one factor to the
same total as M+L), **Flat-JV** (JV's price raised to the same total as M+L). Demand rules use JV's actual price as the
reference: `none` keeps every order; `e1` is elasticity 1; `cons_e1` and `cons_e2` let price rises lose orders and price
cuts win none; `keep_1.00 / 1.10 / 1.25` keep an order only if our price is at most 1.00 / 1.10 / 1.25 × JV's price.

In [4]:
PERIODS = {"Feb-Jun": ["2026-02", "2026-03", "2026-04", "2026-05", "2026-06"], "Jul": ["2026-07"],
           "Feb-Jul": ["2026-02", "2026-03", "2026-04", "2026-05", "2026-06", "2026-07"]}
RULES = {"elasticity=0": "none", "elasticity=1": "e1", "conservative_elasticity=1": "cons_e1",
         "conservative_elasticity=2": "cons_e2", "keep_if_within_0pct": "keep_1.00",
         "keep_if_within_10pct": "keep_1.10", "keep_if_within_25pct": "keep_1.25"}


def strategies(p: pd.DataFrame) -> dict:
    jv, m = p["premiumAmount_USDC"].to_numpy(), p["model_premium_USDC"].to_numpy()
    ml = m * p["loading"].to_numpy()
    return {"JV": jv, "M": m, "M+L": ml, "Flat-M": m * s.flat_factor(ml.sum(), m),
            "Flat-JV": jv * s.flat_factor(ml.sum(), jv)}


scores, intervals, ginis, levels = [], [], [], []
for period, months in PERIODS.items():
    p = panel[panel["month"].isin(months)]
    strat = strategies(p)
    payout, ref = p["realizedProfit_USDC"].to_numpy(), p["premiumAmount_USDC"].to_numpy()
    day = pd.to_datetime(p["day"]).to_numpy("datetime64[D]")  # polars (Maksim's helpers) accepts day resolution
    sc = evaluate_scenarios(strat, payout, ref, day, elasticities=[0.0, 1.0], tolerances=(0.0, 0.10, 0.25),
                            conservative_elasticities=(1.0, 2.0)).to_pandas()
    scores.append(sc.assign(period=period, rule=sc["scenario"].map(RULES)))
    weights = {"none": lambda x: np.ones_like(x), "cons_e2": lambda x: conservative_weight(x, ref, 2.0)}
    for rule, w in weights.items():
        pnl = {k: w(v) * (v - payout) for k, v in strat.items()}
        for a, b in [("M+L", "M"), ("M+L", "Flat-M"), ("M+L", "JV"), ("M", "JV")]:
            intervals.append({"period": period, "rule": rule, "comparison": f"{a} - {b}",
                              **paired_profit_interval(day, pnl[a], pnl[b])})
    for k in ("M", "M+L"):
        ginis.append({"period": period, "strategy": k, **gini_with_interval(ref, strat[k], payout, day)})
    levels += [{"period": period, "strategy": k, "price_vs_jv": v.sum() / ref.sum()} for k, v in strat.items()]

scores = pd.concat(scores, ignore_index=True)
intervals, ginis, levels = pd.DataFrame(intervals), pd.DataFrame(ginis), pd.DataFrame(levels)
profit = scores.pivot_table(index=["period", "strategy"], columns="rule", values="profit")[list(RULES.values())]
profit.round(0)

rule                 none       e1  cons_e1  cons_e2  keep_1.00  keep_1.10  keep_1.25
period  strategy                                                                     
Feb-Jul Flat-JV   70164.0  57866.0  57866.0  47723.0        0.0        0.0    70164.0
        Flat-M    70164.0  58569.0  54559.0  44502.0    16064.0    21110.0    28771.0
        JV        17437.0  17437.0  17437.0  17437.0    17437.0    17437.0    17437.0
        M         62879.0  53864.0  49790.0  41201.0    15401.0    20727.0    28911.0
        M+L       70164.0  58840.0  54333.0  44415.0    17027.0    21471.0    32042.0
Feb-Jun Flat-JV   39816.0  33452.0  33452.0  28105.0        0.0        0.0    39816.0
        Flat-M    39816.0  36042.0  32549.0  27681.0    11926.0    14067.0    19714.0
        JV        13118.0  13118.0  13118.0  13118.0    13118.0    13118.0    13118.0
        M         36373.0  33846.0  30281.0  26100.0    11686.0    14061.0    20613.0
        M+L       39816.0  36130.0  32463.0  27719.0    11988.0    15296.0    23003.0
Jul     Flat-JV   30347.0  24443.0  24443.0  19687.0        0.0        0.0    30347.0
        Flat-M    30347.0  22538.0  22004.0  16812.0     4443.0     7381.0     8958.0
        JV         4319.0   4319.0   4319.0   4319.0     4319.0     4319.0     4319.0
        M         26506.0  20018.0  19509.0  15101.0     3715.0     6666.0     8298.0
        M+L       30347.0  22710.0  21870.0  16696.0     5039.0     6175.0     9039.0

In [5]:
def profit_of(period: str, rule: str, strategy: str) -> float:
    return float(profit.loc[(period, strategy), rule])


profits = {per: {r: {k: profit_of(per, r, k) for k in ("M", "M+L", "Flat-M")} for r in ("none", "cons_e2")}
           for per in ("Jul", "Feb-Jun")}
decision = s.adoption_decision(profits)
print(json.dumps(decision, indent=2))
intervals.round(1)

{
  "c1_july_gain": true,
  "c2_febjun_gain": true,
  "c3_july_beats_flat": false,
  "adopt": false
}


,period,rule,comparison,difference,ci05,ci95,share_of_days_a_better,days
0,Feb-Jun,none,M+L - M,3443.1,2299.8,4613.6,0.4,138
1,Feb-Jun,none,M+L - Flat-M,0.0,-555.6,588.0,0.1,138
2,Feb-Jun,none,M+L - JV,26698.2,16967.4,37215.5,0.6,138
3,Feb-Jun,none,M - JV,23255.1,13954.3,33114.5,0.6,138
4,Feb-Jun,cons_e2,M+L - M,1618.8,793.8,2493.8,0.2,138
5,Feb-Jun,cons_e2,M+L - Flat-M,37.8,-610.1,740.3,0.2,138
6,Feb-Jun,cons_e2,M+L - JV,14600.6,1458.5,30496.4,0.4,138
7,Feb-Jun,cons_e2,M - JV,12981.8,96.8,28591.9,0.4,138
8,Jul,none,M+L - M,3841.5,2943.7,4693.7,0.8,31
9,Jul,none,M+L - Flat-M,0.0,-544.9,505.0,0.5,31


In [6]:
keep = ["share_cheaper_than_current", "median_price_vs_current", "orders_vs_today", "current_premium_retained",
        "loss_ratio", "worst_day", "max_drawdown", "sd_daily_pnl"]
better_not_higher = (scores[scores["rule"].isin(["none", "cons_e2"])]
                     .pivot_table(index=["period", "strategy"], columns="rule", values=keep))
display(levels.pivot(index="strategy", columns="period", values="price_vs_jv").round(3),
        ginis.round(3), better_not_higher.round(3))

period,Feb-Jul,Feb-Jun,Jul
strategy,,,
Flat-JV,1.213,1.190,1.242
Flat-M,1.213,1.190,1.242
JV,1.000,1.000,1.000
M,1.183,1.166,1.206
M+L,1.213,1.190,1.242


,period,strategy,gini,gini_ci05,gini_ci95,share_boot_positive
0,Feb-Jun,M,0.244,0.160,0.314,1.000
1,Feb-Jun,M+L,0.248,0.165,0.317,1.000
2,Jul,M,0.097,0.041,0.148,0.996
3,Jul,M+L,0.115,0.057,0.174,0.999
4,Feb-Jul,M,0.188,0.127,0.247,1.000
5,Feb-Jul,M+L,0.197,0.138,0.254,1.000


current_premium_retained      loss_ratio        max_drawdown            median_price_vs_current        orders_vs_today      sd_daily_pnl           share_cheaper_than_current         \
rule                              cons_e2 none    cons_e2   none      cons_e2       none                 cons_e2   none         cons_e2 none      cons_e2      none                    cons_e2   none   
period  strategy                                                                                                                                                                                        
Feb-Jul Flat-JV                     0.680  1.0      0.767  0.767     6962.096  10235.906                   1.213  1.213           0.680  1.0      953.426  1401.758                      0.000  0.000   
        Flat-M                      0.707  1.0      0.763  0.767     2485.770   9292.214                   0.948  0.948           0.814  1.0      715.016  1265.565                      0.553  0.553   
        JV                          1.000  1.0      0.930  0.930    13030.535  13030.535                   1.000  1.000           1.000  1.0     1308.934  1308.934                      0.000  0.000   
        M                           0.725  1.0      0.782  0.786     2686.686   9673.334                   0.925  0.925           0.827  1.0      721.368  1246.382                      0.575  0.575   
        M+L                         0.708  1.0      0.762  0.767     2470.649   9247.427                   0.936  0.936           0.817  1.0      719.060  1272.417                      0.562  0.562   
Feb-Jun Flat-JV                     0.706  1.0      0.762  0.762     7417.213  10507.839                   1.190  1.190           0.706  1.0      881.130  1248.283                      0.000  0.000   
        Flat-M                      0.732  1.0      0.739  0.762     2347.251   9350.113                   0.995  0.995           0.788  1.0      577.846  1054.719                      0.506  0.506   
        JV                          1.000  1.0      0.907  0.907    12829.104  12829.104                   1.000  1.000           1.000  1.0     1167.765  1167.765                      0.000  0.000   
        M                           0.746  1.0      0.754  0.778     2503.704   9673.334                   0.975  0.975           0.800  1.0      573.003  1043.629                      0.525  0.525   
        M+L                         0.731  1.0      0.737  0.762     2470.649   9247.427                   0.987  0.987           0.790  1.0      582.293  1058.271                      0.512  0.512   
Jul     Flat-JV                     0.649  1.0      0.773  0.773     2365.117   3645.748                   1.242  1.242           0.649  1.0     1211.490  1867.471                      0.000  0.000   
        Flat-M                      0.674  1.0      0.794  0.773     2449.096   3689.477                   0.862  0.862           0.862  1.0     1106.165  1872.777                      0.641  0.641   
        JV                          1.000  1.0      0.960  0.960     5072.863   5072.863                   1.000  1.000           1.000  1.0     1834.509  1834.509                      0.000  0.000   
        M                           0.698  1.0      0.817  0.796     2686.686   3817.995                   0.837  0.837           0.875  1.0     1157.485  1852.135                      0.665  0.665   
        M+L                         0.677  1.0      0.795  0.773     2320.215   3621.777                   0.851  0.851           0.866  1.0     1118.404  1884.157                      0.651  0.651   

                 worst_day            
rule               cons_e2      none  
period  strategy                      
Feb-Jul Flat-JV  -5585.320 -8211.724  
        Flat-M   -2485.770 -7421.243  
        JV       -9629.875 -9629.875  
        M        -2686.686 -7636.313  
        M+L      -2470.649 -7409.499  
Feb-Jun Flat-JV  -5901.417 -8360.437  
        Flat-M   -2347.251 -7453.916  
        

## D. The blind file

The open book before each validation order holds earlier training and validation orders that have not expired.
The main column is Maksim's frozen premium × loading only if the rule passed. The 544 rows with a markup reveal JV's
own price; we compare price levels there (no outcome is used). The exposure cap counts each protection until expiry.

In [7]:
sub, book_valid = s.price_blind_file(DATA, adopted=decision["adopt"])
load = sub["orderflow_loading"]
with_loading = sub["premium_frozen_model_USDC"] * load
print(f"loaded share {np.mean(load > 1):.4f} | mean loading on loaded {load[load > 1].mean():.4f} | "
      f"total: frozen {sub['premium_frozen_model_USDC'].sum():,.2f} -> with loading {with_loading.sum():,.2f} "
      f"({with_loading.sum() / sub['premium_frozen_model_USDC'].sum() - 1:+.2%}) | adopted: {decision['adopt']}")

rev_frozen = s.revealed_price_ratio(valid, sub["premium_frozen_model_USDC"])
rev_loaded = s.revealed_price_ratio(valid, with_loading)
rev_final = s.revealed_price_ratio(valid, sub["recommended_premium_USDC"])
shown = valid["markup_per_BTC_USDC"].notna() & valid["apiBasePrice_per_BTC_USDC"].notna()
revealed_by_duration = pd.DataFrame([
    {"productType": d, "rows": len(g),
     "frozen_vs_jv": s.revealed_price_ratio(g, sub.loc[g.index, "premium_frozen_model_USDC"])["ratio"],
     "final_vs_jv": s.revealed_price_ratio(g, sub.loc[g.index, "recommended_premium_USDC"])["ratio"]}
    for d, g in valid[shown].groupby("productType")])
print("revealed JV price, 544 rows:", {k: round(v["ratio"], 4) for k, v in
                                       {"frozen": rev_frozen, "with_loading": rev_loaded, "final": rev_final}.items()})
revealed_by_duration.round(3)

loaded share 0.2491 | mean loading on loaded 1.0722 | total: frozen 205,823.94 -> with loading 211,552.86 (+2.78%) | adopted: False
revealed JV price, 544 rows: {'frozen': 0.9572, 'with_loading': 0.9784, 'final': 0.9572}


,productType,rows,frozen_vs_jv,final_vs_jv
0,1800,106,0.670,0.670
1,3600,16,1.062,1.062
2,7200,112,1.021,1.021
3,14400,243,0.932,0.932
4,28800,65,1.064,1.064
5,43200,2,1.231,1.231


In [8]:
start = valid["createdAt"].min()
cap_orders = pd.concat([train[train["expirationDate"] > start].assign(source="training"),
                        valid.assign(source="validation")], ignore_index=True)
cap_orders["notional_USDC"] = cap_orders["quantity_BTC"] * cap_orders["strikePrice"]
accepted = s.exposure_cap_accepts(cap_orders)[(cap_orders["source"] == "validation").to_numpy()]
premium = sub["recommended_premium_USDC"]
cap_paused_orders = float(1 - accepted.mean())
cap_paused_premium = float(premium[~accepted].sum() / premium.sum())
cap_by_month = (pd.DataFrame({"month": valid["createdAt"].dt.strftime("%Y-%m"), "paused": ~accepted,
                              "premium": premium})
                .groupby("month").apply(lambda g: pd.Series({"orders": len(g), "paused_share": g["paused"].mean(),
                                                            "premium_paused_share": g.loc[g["paused"], "premium"].sum()
                                                            / g["premium"].sum()}), include_groups=False))
print(f"cap {s.CAP_PER_DIRECTION_USDC:,} USDC per direction: pauses {cap_paused_orders:.1%} of orders, "
      f"{cap_paused_premium:.1%} of premium")
cap_by_month.round(3)

cap 250,000 USDC per direction: pauses 9.4% of orders, 29.9% of premium


,orders,paused_share,premium_paused_share
month,,,
2026-07,118.0,0.000,0.000
2026-08,2129.0,0.073,0.305
2026-09,1340.0,0.135,0.304
2026-10,568.0,0.099,0.263


In [9]:
rebuilt = pd.read_csv(PATHS["rebuilt_base"])
assert len(rebuilt) == len(sub)
ratio = sub["recommended_premium_USDC"] / (rebuilt["anchor_per_BTC_USDC"] * sub["quantity_BTC"])
rebuilt_check = (pd.DataFrame({"anchor_source": rebuilt["anchor_source"], "premium_over_anchor": ratio})
                 .groupby("anchor_source")["premium_over_anchor"].describe(percentiles=[0.25, 0.5, 0.75]))
print(f"Subham's hybrid total {rebuilt['premiumAmount_USDC'].sum():,.0f} vs final {premium.sum():,.0f} USDC")
rebuilt_check.round(3)

Subham's hybrid total 216,764 vs final 205,824 USDC


,count,mean,std,min,25%,50%,75%,max
anchor_source,,,,,,,,
jasper,1960.0,1.329,0.395,1.000,1.000,1.182,1.553,2.587
rebuilt,2195.0,1.781,0.601,0.494,1.322,1.777,2.079,4.800


## E. Moneyness

Every order in the data is at the money, so moneyness cannot be tested on outcomes. Maksim's pricer handles it by
construction: the same Black-Scholes formula prices a strike away from spot. The table uses the median expected move
of each product length in the blind file.

In [10]:
DISTANCES = [0.0, 0.0025, 0.005, 0.01, 0.02]
moneyness = pd.concat([
    s.moneyness_table(g["expected_abs_move_bps"].median() / 1e4, float(g["strikePrice"].median()), DISTANCES)
    .assign(productType=d, median_expected_move_bps=g["expected_abs_move_bps"].median())
    for d, g in frozen.groupby("productType")], ignore_index=True)
moneyness.pivot_table(index=["productType", "side"], columns="otm_distance", values="vs_at_the_money").round(3)

otm_distance      0.0000  0.0025  0.0050  0.0100  0.0200
productType side                                        
1800        call     1.0   0.181   0.014   0.000   0.000
            put      1.0   0.180   0.014   0.000   0.000
3600        call     1.0   0.511   0.225   0.026   0.000
            put      1.0   0.508   0.222   0.025   0.000
7200        call     1.0   0.580   0.305   0.060   0.001
            put      1.0   0.578   0.301   0.058   0.000
14400       call     1.0   0.664   0.417   0.136   0.006
            put      1.0   0.662   0.413   0.132   0.005
28800       call     1.0   0.718   0.496   0.210   0.022
            put      1.0   0.715   0.491   0.204   0.019
43200       call     1.0   0.750   0.546   0.265   0.042
            put      1.0   0.747   0.542   0.258   0.038
86400       call     1.0   0.827   0.676   0.435   0.152
            put      1.0   0.825   0.671   0.426   0.142

## F. Outputs

In [11]:
sub_path = OUT / "final_submission.csv"
sub.to_csv(sub_path, index=False, lineterminator="\n")
summary = {
    "adopted": decision["adopt"], "decision": decision,
    "frozen_sha256_matches": frozen_ok, "frozen_sha256": frozen_record["sha256"],
    "submission_sha256": sha256(sub_path), "rows": int(len(sub)),
    "premium_total_frozen": float(sub["premium_frozen_model_USDC"].sum()),
    "premium_total_with_loading": float(with_loading.sum()),
    "premium_total_final": float(sub["recommended_premium_USDC"].sum()),
    "loaded_share": float(np.mean(load > 1)), "mean_load_on_loaded": float(load[load > 1].mean()),
    "revealed_rows": rev_final["rows"], "revealed_ratio_frozen": rev_frozen["ratio"],
    "revealed_ratio_with_loading": rev_loaded["ratio"], "revealed_ratio_final": rev_final["ratio"],
    "cap_paused_share_orders": cap_paused_orders, "cap_paused_share_premium": cap_paused_premium,
    "settings": {"k_loading": s.K_LOADING, "min_book_btc": s.MIN_BOOK_BTC, "margin": s.MARGIN,
                 "cap_per_direction_usdc": s.CAP_PER_DIRECTION_USDC},
    "repo_commit_at_run": commit,
    "inputs_sha256": {k: sha256(v) for k, v in PATHS.items()},
}
(OUT / "summary.json").write_text(json.dumps(summary, indent=2) + "\n")
tables = {"training_scores": scores, "training_intervals": intervals, "training_gini": ginis,
          "training_price_levels": levels, "training_loading_by_month": loading_by_month.reset_index(),
          "training_book_classes": book_classes.reset_index(), "blind_revealed_by_duration": revealed_by_duration,
          "blind_cap_by_month": cap_by_month.reset_index(), "blind_rebuilt_base_check": rebuilt_check.reset_index(),
          "moneyness": moneyness}
for name, t in tables.items():
    t.to_csv(OUT / f"{name}.csv", index=False, lineterminator="\n")
print("adopted:", summary["adopted"], "| final total:", f"{summary['premium_total_final']:,.2f}",
      "| sha256:", summary["submission_sha256"][:16])

adopted: False | final total: 205,823.94 | sha256: 810eae3a1c007c94
